In [54]:
import pandas as pd
import importlib
import funciones as func #Tener funciones.py en mismo directorio. Tiene las funciones usadas para procesar un df
importlib.reload(func)

<module 'funciones' from 'c:\\Users\\dafyd\\Documents\\Escuela\\2025\\semestre 1\\TD6\\TP2\\last_day\\funciones.py'>

In [55]:
data = pd.read_csv('../competition_data.csv')

In [56]:
import csv
f = open('../uri_to_duration.csv', 'r')
diccionario = {}
for line in csv.DictReader(f):
    diccionario[line['uri']] = float(line['duration'])
f.close()
diccionario

{'spotify:track:3jFfr89lnSmb4QBtfG8JBP': 187120.0,
 'spotify:track:09TTeexnlKewZdjOak2sV2': 292573.0,
 'spotify:track:0G3fbPbE1vGeABDEZF0jeG': 495400.0,
 'spotify:track:1NXvuBAq48QrxRFQZVmORQ': 430960.0,
 'spotify:track:2gYJY0sIx1ErgTIha2nPRg': 214848.0,
 'spotify:track:0njXuAkJVSpptzKniMBtIZ': 405499.0,
 'spotify:track:7Kszjzps0xbQXyo1pO4KfE': 331360.0,
 'spotify:track:39KmBOGkD1ztCbVeo2uspA': 277560.0,
 'spotify:track:0UAEHlFR79k9CJvknSGUNf': 215040.0,
 'spotify:track:1sWeSMifj6Z6kZyI6z3bRc': 171040.0,
 'spotify:track:0nLiqZ6A27jJri2VCalIUs': 388266.0,
 'spotify:track:1Ig4pCPanhzXl7C86MmBUc': 161133.0,
 'spotify:track:0WtDGnWL2KrMCk0mI1Gpwz': 326933.0,
 'spotify:track:326HJ8o8XJff8dSZOwe4GS': 288586.0,
 'spotify:track:1bSpwPhAxZwlR2enJJsv7U': 229120.0,
 'spotify:track:5luOvrlnzfvJQdQjrScVj4': 249500.0,
 'spotify:track:1SJtlNRJDeYHioymcvsqev': 216391.0,
 'spotify:track:06pjQCIybMiGrgtHODGa2m': 225906.0,
 'spotify:track:6eJlEcRmeyQfTlDQBDyqkW': 156680.0,
 'spotify:track:6A64S6NPQwA5KIQ

In [57]:
data = func.sort_by_ts(data)

In [ ]:
# Toma aleatoriamente el 80 % de las filas
train = data.sample(frac=0.7, random_state=831).copy()

# El resto (20 %) lo podés obtener excluyendo esos índices
validation   = data.drop(train.index).copy()

In [59]:
data = func.procesar_df(data, diccionario)
train = func.procesar_df(train, diccionario)
validation = func.procesar_df_2(validation, diccionario)

In [60]:
validation = validation.merge(
    data[['Unnamed: 0', 'is_early_finish', 'racha_skips_prev', 'racha_tipo']],
    on='Unnamed: 0',
    how='left'
)

In [61]:
validation = validation[data.columns]

In [ ]:
import xgboost as xgb
from sklearn.metrics import roc_auc_score

drop_cols = [
    'Unnamed: 0','ts', 'platform', 'reason_start',
    'master_metadata_track_name',
    'master_metadata_album_artist_name',
    'master_metadata_album_album_name',
    'spotify_track_uri','platform', 'TARGET'
]
aucs = {}
for l in [0.1, 1, 5, 10]:
    for g in [0, 0.1, 0.5, 1]:
        for e in [0.01, 0.05, 0.1, 0.3]:
            clf_xgb = xgb.XGBClassifier(objective = 'binary:logistic',
                                        seed = 831,
                                        eval_metric = 'auc',
                                        early_stopping_rounds = 100,
                                        learning_rate = e,
                                        gamma = g,
                                        reg_lambda=l)
            clf_xgb.fit(train.drop(columns=drop_cols),
                            train['TARGET'],
                            eval_set=[(validation.drop(columns=drop_cols), validation['TARGET'])],
                            verbose=False
                            )

            best = clf_xgb.best_iteration
            prediccion_val = clf_xgb.predict_proba(validation.drop(columns=drop_cols))[:, 1]
            auc = roc_auc_score(validation['TARGET'], prediccion_val)
            aucs[(l, g, e)] = auc

In [63]:
best = None
best_auc = 0
for clave in aucs.keys():
    if best is None or aucs[clave] > aucs[best]:
        best = clave
        best_auc = aucs[clave]

print(f"Mejor combinación: {best} con AUC-ROC: {best_auc:.4f}")

Mejor combinación: (5, 0.5, 0.3) con AUC-ROC: 0.9675
